# model-train-eval-toggle-around-sample — worked example 2: Exception-safe eval_mode context manager

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `model-train-eval-toggle-around-sample`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Wrapping the eval/no_grad toggle in a `@contextmanager` with `try/finally` guarantees the prior `model.training` state is restored even if the body raises. It captures `was_training`, enters eval + no_grad, yields, and restores in `finally`.

## Worked solution

We build a reusable, exception-safe eval block.

1. **Capture prior state.** `was_training = model.training` so we can restore exactly, whether the model started in train or eval.
2. **Enter eval + no_grad.** `model.eval()` then `with t.no_grad(): yield model`.
3. **finally restore.** In the `finally` clause, if `was_training` we call `model.train()`. This runs even when the body raises — the whole point of the wrapper.
4. **Two scenarios.** We run a normal block and a raising block; in both the model ends in its original mode and BN stats are untouched.

The demo runs both a clean and a raising `with eval_mode(model):` block and prints that training mode is correctly restored each time.

In [ ]:
import torch as t
import torch.nn as nn
import contextlib

t.manual_seed(1)

model = nn.Sequential(nn.Linear(3, 6), nn.BatchNorm1d(6))
model.train()

@contextlib.contextmanager
def eval_mode(model):
    was_training = model.training
    model.eval()
    try:
        with t.no_grad():
            yield model
    finally:
        if was_training:
            model.train()

with eval_mode(model):
    assert not model.training
print('restored after clean block:', model.training)

try:
    with eval_mode(model):
        raise ValueError('boom')
except ValueError:
    pass
print('restored after raising block:', model.training)